<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

In [ ]:
%matplotlib inline
import time
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython import display
import cv2
import copy
from tqdm.contrib import tzip

# Filtre à particules

Le filtre particulaire permet d'estimer l'état d'un robot en suivant un ensemble d'états probables et en les mettant à jour à mesure que de nouvelles mesures sont effectuées. Ceci est réalisé grâce à un ensemble de particules, où chaque particule $i$ représente une hypothèse de l'état $x_t^i$ avec son poids associé $w_t^i$. Ainsi, par exemple, dans le contexte de l'estimation d'état (par exemple, la localisation), chaque particule représente la probabilité que le robot se trouve dans cet état particulier.

L'algorithme est le suivant. Premièrement, nous initialisons le filtre particulaire en échantillonnant $M$ particules. Généralement, l'ensemble initial de particules peut être échantillonné uniformément parmi tous les états possibles (échantillonnage uniforme). Étant donné un signal de commande $u_t$, nous déplaçons toutes les particules (c'est-à-dire l'état associé à chaque particule) en suivant le modèle de mouvement. Ensuite, nous effectuons une mesure $z_t$ (obtenue grâce à notre capteur) et mettons à jour le poids de chaque particule en fonction de la probabilité de mesurer $z_t$ dans l'état correspondant (par exemple, $p(z_t | x_i)$). Nous rééchantillonnons ensuite toutes les particules, non pas uniformément, mais en fonction de leur poids respectif (les particules ayant un poids élevé ont plus de chances d'être échantillonnées). Ainsi, les particules les plus importantes seront échantillonnées plus fréquemment, tandis que celles ayant un poids faible risquent de ne pas l'être du tout. Différentes méthodes permettent d'obtenir l'état prédit (c'est-à-dire filtré). Dans ce notebook, nous utiliserons l'approche la plus simple en calculant la moyenne des particules rééchantillonnées.

Nous répétons ensuite ce processus à chaque nouvelle valeur de $u_t$ et $z_t$. 

L'algorithme présenté ci-dessus est l'algorithme de base du filtre particulaire. Il présente certains défis, et différentes variantes tentent de les résoudre. Par exemple, si nous implémentons cet algorithme, les performances seront fortement affectées par l'ensemble initial de particules, car lors du rééchantillonnage, nous ne sélectionnons que des particules parmi cet ensemble initial. Il est donc possible que nous n'ayons aucune particule proche de l'état correct. Ce problème est parfois appelé « appauvrissement en particules ». Pour atténuer ce problème, lors du rééchantillonnage, au lieu de sélectionner toutes les particules en fonction de leur poids, nous pouvons sélectionner aléatoirement un petit pourcentage de particules parmi toutes les particules possibles.

Prenons un exemple concret.

## Exemple : Localisation de robots avec filtre particulaire

Considérons un robot se déplaçant dans une pièce sans obstacles. Ce robot est équipé de deux capteurs mesurant la distance entre lui et les murs, ce qui lui permet de déterminer sa position (positions $x$ et $y$) dans la pièce. Par souci de simplification, supposons que les capteurs fournissent directement une mesure bruitée de la position $(x,y)$ dans la pièce.

Supposons que l'état du robot soit sa position $x$ et $y$ dans la pièce, et que les entrées de commande soient sa vitesse dans chaque direction, $v_x$ et $v_y$. Le robot est initialisé à $(x,y) = (0,0)$ et se déplace en appliquant des entrées de commande constantes $v_x = v_y = 0,1$ pendant 100 itérations. À chaque itération, après l'application d'un signal de commande, le robot effectue une mesure à l'aide des capteurs afin de déterminer sa position.

Nous supposons le modèle de mouvement suivant :

$$
x_t = x_{t-1} + u_t + w_t,
$$

où $w_t \sim \mathcal{N}([0, 0]^T,\sigma_R\mathcal{I}_2)$ ($\mathcal{I}_2$ étant la matrice identité 2\times2$). Contrairement au filtre de Kalman, il est important de noter que le modèle n'est pas contraint d'être linéaire ni de présenter un bruit additif ou gaussien. Ce choix de définition est motivé par un souci de simplicité.

Supposons également que le fabricant du capteur nous ait fourni des informations sur ses performances. Les mesures ont une moyenne nulle et un écart-type $\sigma$. Le modèle de mesure est alors le suivant :
$$ 
z_t = x_t + n_t
$$

où $n_t \sim \mathcal{N}([0, 0]^T,\sigma_Q\mathcal{I}_2$). Il est important de noter que cette structure n'est pas obligatoire pour un filtre particulaire, mais nous l'adoptons ici par souci de simplicité.

Nous utiliserons le filtre particulaire pour améliorer notre estimation de la position du robot à chaque pas de temps.

### Comprendre le problème

Pour comprendre le problème, considérons la trajectoire idéale et quelques mesures possibles que nous obtiendrions à l'aide des capteurs disponibles, conformément à leurs spécifications. En pratique, nous obtenons des mesures de nos capteurs. Mais ici, pour simuler ces mesures, nous utiliserons l'état réel à chaque instant et ajouterons un bruit gaussien suivant une loi normale $\mathcal{N}(0, 0.5)$.

In [ ]:
# state = [x_pos, y_pos]
num_data = 100  # il s'agit du nombre d'étapes temporelles que nous allons simuler pour
ground_truth_x = np.linspace(0, 10, num=num_data + 1)
ground_truth_y = ground_truth_x.copy()  # x = y

# Simuler les mesures des capteurs
measurement_noise_x_var = 0.5  # ce sont les covariances de mesure "vrai"
measurement_noise_y_var = 0.5
noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var, size=num_data - 1)
noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var, size=num_data - 1)
measurement_x = np.linspace(10 / num_data, 10, num=num_data - 1) + noise_x
measurement_y = np.linspace(10 / num_data, 10, num=num_data - 1) + noise_y

# Comparer les données de terrain et les mesures
plt.figure(figsize=(8, 8))
plt.plot(ground_truth_x, ground_truth_y)
plt.plot(measurement_x, measurement_y)
plt.xlabel("x position")
plt.ylabel("y position")
plt.legend(["ground truth trajectory", "localization measurements from noisy sensor"])
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Comme on peut le constater, le capteur émet beaucoup de bruit.

### Mise en œuvre du filtre particulaire

Commençons par créer une classe Particle. Chaque particule doit posséder un état et un poids. Nous avons également besoin d'une méthode permettant à une particule de se déplacer à partir d'un signal de contrôle $u_t$ (c'est-à-dire, prédire son déplacement) et d'une méthode pour mettre à jour son poids à partir d'une mesure $z_t$. Plus tard, nous aurons également besoin d'une fonction pour sélectionner une particule dans un ensemble, en tenant compte de son poids.

In [ ]:
# Définissons les covariances de nos modèles de mouvement et de mesure 
# (nous pouvons les ajuster et observer les résultats).

import random

sigma_R = 0.01
sigma_Q = 0.5

In [ ]:
class Particle:
    def __init__(self, x, y, w):
        self.state = np.array([x, y])
        self.weight = w

    def predict(self, u_t):
        # Il faut échantillonner la particule à l'aide de la fonction de prédiction (cela nécessitera un échantillonnage gaussien).
        # Par exemple, vous pouvez utiliser la fonction scipy.stats.multivariate_normal(...).pdf(...) – voir la documentation en ligne.             
        self.state = self.state  # TODO 

    def update(self, z_t):
        # Nous devons maintenant calculer le poids en fonction de la probabilité de mesure
        self.weight = 1  # TODO 


def sample_particle(particles):
    # TODO Écrivez cette fonction pour échantillonner une particule en fonction des poids.
    # Il peut être nécessaire de normaliser les pondérations pour obtenir une fonction de distribution de probabilité valide (c'est-à-dire que la somme des pondérations est égale à 1).
    return particles[0]

Ensuite, nous implémentons le filtre particulaire et l'exécutons à chaque étape temporelle.

In [ ]:
num_particles = 100  # Pourrait être accordé
# alpha est le pourcentage des particules que nous allons rééchantillonner à partir de 
# l'ensemble de particules pondéré (au lieu d'être initialisées aléatoirement)
alpha = 0.95 

filtered_xs = []
filtered_ys = []
measurements = []
particles_over_time = []

# initialiser l'ensemble de particules
particles = []


def initialize_particle():
    x = np.random.uniform(0, 10)
    y = np.random.uniform(0, 10)
    w = 1
    return Particle(x, y, w)


for i in range(num_particles):
    particles.append(initialize_particle())

# Dans cet exemple particulier, nous supposons un signal de commande constant.
u_t = np.array([10.0 / num_data, 10.0 / num_data])

# Exécuter le filtre particulaire à chaque étape temporelle
for i in range(num_data):
    # Étant donné u_t, déplacez toutes les particules en suivant le modèle de mouvement.
    for p in particles:
        p.predict(u_t)

    # obtenir la mesure z_t (en réalité, obtenir cette mesure à partir de notre capteur)
    measurement_noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var)
    measurement_noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var)
    measurement_x_new = ground_truth_x[i + 1] + measurement_noise_x
    measurement_y_new = ground_truth_x[i + 1] + measurement_noise_y
    z_t = np.array([measurement_x_new, measurement_y_new])
    measurements.append([measurement_x_new, measurement_y_new])

    # étant donné z_t, mettre à jour les poids des particules.
    for p in particles:
        p.update(z_t)

    # Stocker les particules rééchantillonnées afin de pouvoir les représenter graphiquement ultérieurement.
    particles_over_time.append(copy.deepcopy(particles))

    new_particles = []
    for i in range(int(num_particles * alpha)):
        new_particle = copy.deepcopy(sample_particle(particles))
        new_particle.weight = 1
        new_particles.append(new_particle)
    for i in range(int(num_particles * (1 - alpha))):
        new_particles.append(initialize_particle())

    particles = new_particles

    # obtenir une estimation de l'état en prenant la moyenne des particules rééchantillonnées (à l'exclusion de celles échantillonnées aléatoirement)
    xs = []
    ys = []
    for p in particles[: int(num_particles)]:
        xs.append(p.state[0])
        ys.append(p.state[1])
    estimated_x = np.mean(xs)
    estimated_y = np.mean(ys)

    # Stocker l'état filtré pour pouvoir le représenter graphiquement ultérieurement.
    filtered_xs.append(estimated_x)
    filtered_ys.append(estimated_y)

measurements = np.array(measurements)

Visualisons l'évolution des particules au fil du temps afin de mieux comprendre le fonctionnement du filtre particulaire.

In [ ]:
ground_truth = np.stack((ground_truth_x, ground_truth_y), axis=1)
filtered_states = np.stack((filtered_xs, filtered_ys), axis=1)

In [ ]:
# Une fenêtre contextuelle peut apparaître avec le texte "Widget require us to download supporting files from a 3rd party". Cliquez sur le bouton 'Enable Downloads'.

plots = []

t = 0
for set_of_ps, true_state, filtered_state, z_t in tzip(
    particles_over_time, ground_truth[1:], filtered_states, measurements
):
    fig = plt.figure()
    plt.scatter(true_state[0], true_state[1], color="blue", s=75)
    plt.scatter(filtered_state[0], filtered_state[1], color="green", s=75)
    plt.scatter(z_t[0], z_t[1], color="orange", s=75)

    for p in set_of_ps:
        plt.scatter(p.state[0], p.state[1], color="red", s=p.weight * 50)
    plt.title(
        "time=%d, true (x,y)=(%.2f, %.2f) \n filtered (x,y)=(%.2f, %.2f)"
        % (t + 1, true_state[0], true_state[1], filtered_state[0], filtered_state[1])
    )
    plt.xlabel("x position")
    plt.ylabel("y position")
    plt.xlim(0, 15)
    plt.ylim(0, 15)
    plt.gca().set_aspect("equal", adjustable="box")

    blue_patch = patches.Patch(color="blue", label="true state")
    red_patch = patches.Patch(color="red", label="weighted particles")
    green_patch = patches.Patch(color="green", label="filtered state")
    orange_patch = patches.Patch(color="orange", label="noisy measurement")
    plt.legend(
        handles=[blue_patch, red_patch, green_patch, orange_patch],
        loc="upper left",
        fontsize=8,
    )

    fig.canvas.draw()
    plot_img = np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")
    plot_img = plot_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plots.append(plot_img)
    t += 1
    plt.clf()
    plt.close()

In [ ]:
for im in plots:
    plt.figure(figsize=(14, 14))
    plt.imshow(im)
    plt.axis("off")
    display.display(plt.gcf())
    display.clear_output(wait=True)
    time.sleep(0.1)
    plt.clf()
    plt.close()

In [ ]:
# Exécutez ceci si vous souhaitez enregistrer l'animation de l'intrigue au format GIF.

# import imageio
# imageio.mimsave('pf.gif', plots)

Comme on peut le constater, les particules (points rouges) les plus proches de la position du robot (points bleus) grossissent au fil du temps (leur poids augmente) à mesure que le robot se déplace. L'état prédit par le filtre particulaire est visualisé par le point vert, tandis que la mesure du capteur est représentée par le point orange.

Remarquez que les particules mettent un certain temps à converger vers l'état réel. Ceci s'explique par les performances médiocres observées aux premiers instants. En effet, les particules sont initialement générées aléatoirement, ce qui ne permet pas une estimation précise tant que nous n'avons pas suffisamment de mesures. Représentons graphiquement l'estimation de l'état au fil du temps ci-dessous.

In [ ]:
# Représentons graphiquement les résultats
plt.figure(figsize=(8, 8))
plt.plot(ground_truth[:, 0], ground_truth[:, 1])
plt.plot(measurements[:, 0], measurements[:, 1])
plt.plot(filtered_states[:, 0], filtered_states[:, 1])
plt.xlabel("x position")
plt.ylabel("y position")
plt.legend(
    ["ground truth trajectory", "noisy measurements", "particle filter estimate"]
)
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Le filtre semble fonctionner correctement, car la ligne verte est proche de la droite $x=y$. Comment pouvons-nous améliorer encore ses performances ?

Bien qu'un plus grand nombre de particules permette une meilleure approximation de la distribution réelle que nous cherchons à estimer, cela augmente le coût des calculs, ce qui peut constituer une limitation selon le matériel utilisé.

Vous pouvez expérimenter avec des paramètres tels que le nombre de particules ou le paramètre $\alpha$ pour tenter d'améliorer encore les résultats.

Nous pouvons maintenant passer au dernier notebook sur le [filtre de Kalman étendu](../03-histogram-filter/histogram_filter.ipynb).